# Chapter 17 Companion Notebook
**Build Your First LLM — Chapter 17: Deployment Options**

This notebook bundles the runnable code examples from Chapter 17. Modal deployment runs from your terminal, not Colab, but this notebook explains each step.

- Installs: modal
- Data: inline examples; no external files needed
- Runtime: Most Modal commands run in terminal, not notebook

In [ ]:
# ===== SETUP =====
# Install Modal
!pip install -q modal

print('Modal installed!')
print('IMPORTANT: Run "modal setup" in your terminal to authenticate.')
print('This opens a browser window for login.')

## Section 17.1: Why Modal?

**What is Serverless?**

| Traditional Server | Serverless (Modal) |
|-------------------|-------------------|
| You rent a computer 24/7 | Code runs only when needed |
| You pay even when idle | You pay per second of compute |
| You manage updates, security | Platform handles everything |
| Fixed capacity | Automatic scaling |

**Analogy:** Traditional hosting is like owning a car. Serverless is like using a taxi—you pay only when you ride.

**Why Modal for LLMs?**
- GPU support built-in (T4 to H100)
- $30/month free credits (~50 hours T4 GPU time)
- Python-native (no Docker, no YAML)
- Scales to zero (no charges when idle)

## Section 17.2: Hello Modal

The simplest Modal app. Save this as `hello.py` and run with `modal run hello.py`:

In [ ]:
# Save this as hello.py
hello_modal_code = '''
import modal

app = modal.App("hello-world")

@app.function()
def hello(name: str) -> str:
    return f"Hello, {name}!"

@app.local_entrypoint()
def main():
    result = hello.remote("World")
    print(result)
'''

print("Save this code as 'hello.py':")
print(hello_modal_code)
print("\nThen run: modal run hello.py")

**What just happened?**

1. `modal.App()` creates your application
2. `@app.function()` marks code to run in Modal's cloud
3. `hello.remote()` calls the function remotely
4. Modal spun up a container, ran your code, returned the result

## Section 17.3: Adding GPU Support

One parameter adds GPU support:

In [ ]:
# Save this as gpu_test.py
gpu_test_code = '''
import modal

app = modal.App("gpu-test")

@app.function(gpu="T4")  # That's it!
def check_gpu():
    import torch
    if torch.cuda.is_available():
        device = torch.cuda.get_device_name(0)
        return f"GPU available: {device}"
    return "No GPU found"

@app.local_entrypoint()
def main():
    print(check_gpu.remote())
'''

print("Save this code as 'gpu_test.py':")
print(gpu_test_code)
print("\nThen run: modal run gpu_test.py")
print("\nExpected output: GPU available: Tesla T4")

**One parameter.** No CUDA installation, no driver management, no Docker images. Modal handles the entire GPU stack.

## Section 17.4: Installing Dependencies

Your LLM needs libraries. Modal lets you define your container environment in Python:

In [ ]:
# Save this as with_deps.py
with_deps_code = '''
import modal

# Define the container image
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "torch",
    "transformers",
    "accelerate",
)

app = modal.App("with-deps")

@app.function(image=image, gpu="T4")
def generate_text():
    from transformers import pipeline
    
    generator = pipeline("text-generation", model="gpt2", device=0)
    result = generator("The meaning of life is", max_length=50)
    return result[0]["generated_text"]

@app.local_entrypoint()
def main():
    print(generate_text.remote())
'''

print("Save this code as 'with_deps.py':")
print(with_deps_code)
print("\nThen run: modal run with_deps.py")

The `image` parameter tells Modal what to install. First run takes longer (building the image), but subsequent runs reuse the cached image.

## Section 17.5: Web Endpoints with FastAPI

Create a web endpoint that anyone can access. This is where Chapter 16's FastAPI knowledge pays off:

In [ ]:
# Save this as simple_api.py
simple_api_code = '''
import modal
from fastapi import FastAPI

app = modal.App("my-api")
web_app = FastAPI()

@web_app.get("/health")
def health():
    return {"status": "healthy"}

@web_app.get("/hello/{name}")
def hello(name: str):
    return {"message": f"Hello, {name}!"}

@app.function()
@modal.asgi_app()
def serve():
    return web_app
'''

print("Save this code as 'simple_api.py':")
print(simple_api_code)
print("\nThen run: modal deploy simple_api.py")
print("\nYou'll get a URL like: https://your-workspace--my-api-serve.modal.run")

**Your API is live!** Open that URL in your browser. Add `/health` to the path:

```
https://your-workspace--my-api-serve.modal.run/health
```

You should see: `{"status": "healthy"}`

## Section 17.6: The Complete LLM Service

Here's a production-ready LLM deployment:

In [ ]:
# Save this as llm_service.py
llm_service_code = '''
import modal
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import os

# Define the container image with LLM dependencies
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi",
    "vllm==0.6.4",  # Pin version for reproducibility
    "torch",
)

app = modal.App("my-llm-service")
web_app = FastAPI(title="My LLM API", version="1.0.0")

# Request/Response models (familiar from Chapter 16)
class ChatRequest(BaseModel):
    message: str
    max_tokens: int = 256

class ChatResponse(BaseModel):
    response: str

# Global model reference (loaded once per container)
_model = None

def get_model():
    """Load model once, reuse for all requests."""
    global _model
    if _model is None:
        from vllm import LLM
        _model = LLM(
            model="Qwen/Qwen2.5-1.5B-Instruct",
            trust_remote_code=True,
        )
    return _model

@web_app.get("/health")
def health():
    """Health check endpoint (Chapter 16 callback)."""
    return {"status": "healthy", "model": "Qwen2.5-1.5B-Instruct"}

@web_app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):
    """Generate a response to a message."""
    try:
        model = get_model()
        from vllm import SamplingParams
        
        params = SamplingParams(max_tokens=request.max_tokens)
        outputs = model.generate([request.message], params)
        response_text = outputs[0].outputs[0].text
        
        return ChatResponse(response=response_text)
    except Exception as e:
        raise HTTPException(500, f"Generation failed: {str(e)}")

@app.function(
    image=image,
    gpu="T4",
    timeout=300,
    scaledown_window=300,  # Keep warm for 5 minutes
)
@modal.concurrent(max_inputs=10)  # Handle multiple requests per container
@modal.asgi_app()
def serve():
    return web_app
'''

print("Save this code as 'llm_service.py':")
print(llm_service_code)

In [ ]:
print("Deploy with: modal deploy llm_service.py")
print("\nFirst deployment takes a few minutes (downloading the model).")
print("Subsequent deployments are fast.")
print("\nTest with:")
print('curl https://your-url.modal.run/health')
print('curl -X POST https://your-url.modal.run/chat -H "Content-Type: application/json" -d \'{"message": "What is Python?"}\'')

## Section 17.7: Secrets and Volumes

### Creating Secrets

```bash
# Create a secret from command line
modal secret create my-secrets API_KEY=your_secret_key
```

### Using Secrets in Code

In [ ]:
# Using secrets in your code
secrets_example = '''
@app.function(secrets=[modal.Secret.from_name("my-secrets")])
def with_secrets():
    import os
    api_key = os.environ["API_KEY"]
    # Use the key securely...
'''

print("Using secrets:")
print(secrets_example)

In [ ]:
# Using volumes to cache models
volumes_example = '''
# Create a volume for model cache
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)

@app.function(
    gpu="T4",
    volumes={"/root/.cache/huggingface": model_cache},
)
def with_cache():
    # Models download once, then cached in volume
    pass
'''

print("Using volumes to cache models:")
print(volumes_example)
print("\nFirst request downloads the model. Subsequent requests use the cache.")

## Section 17.8: GPU Options

| Model Size | Recommended GPU | Cost/Hour |
|------------|-----------------|----------|
| < 3B params | T4 (16GB) | $0.59 |
| 3-8B params | A10G (24GB) | $1.10 |
| 8-30B params | A100-40GB | $2.50 |
| 30B+ params | A100-80GB / H100 | $4-8 |

**Start with T4.** Upgrade only when needed.

In [ ]:
# GPU selection examples
print("GPU selection is one parameter:")
print()
print('@app.function(gpu="T4")      # Budget option')
print('@app.function(gpu="A10G")    # Mid-range')
print('@app.function(gpu="A100")    # High performance')
print('@app.function(gpu="H100")    # Maximum power')
print('@app.function(gpu="A100:2")  # Two A100s')

## Section 17.9: Managing Your Deployment

### Viewing Logs

```bash
modal app logs my-llm-service
```

Or use the dashboard at [modal.com](https://modal.com).

### Updating Your App

```bash
# Just redeploy
modal deploy llm_service.py
```

No downtime. Modal handles rolling updates automatically.

### Cost Control

- Use `scaledown_window` to shut down idle containers
- Start with T4, upgrade only when needed
- Monitor spending in the Modal dashboard

## Summary

You've learned how to:

1. **Deploy functions** to Modal with `@app.function()`
2. **Add GPU support** with a single parameter: `gpu="T4"`
3. **Install dependencies** using `modal.Image`
4. **Create web endpoints** with FastAPI + `@modal.asgi_app()`
5. **Store secrets** securely with `modal.Secret`
6. **Cache models** with `modal.Volume`
7. **Control costs** with idle timeouts and monitoring

**Your LLM is no longer trapped on your laptop.** It's accessible to anyone, anywhere, with GPU acceleration.

**Commands to remember:**
```bash
pip install modal      # Install
modal setup            # Authenticate
modal run file.py      # Run once
modal deploy file.py   # Deploy permanently
modal app logs name    # View logs
```